In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


load_dotenv()


model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),  # pyright: ignore[reportArgumentType]
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),  # pyright: ignore[reportArgumentType]
)

In [2]:
response = model.invoke("hi")
response.pretty_print()

================================== Ai Message ==================================

Hello! How can I help you today? 😊


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatOpenAI(
    model="qwen3-32b",
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),  # pyright: ignore[reportArgumentType]
)
advanced_model = ChatOpenAI(
    model="qwen3-max",
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),  # pyright: ignore[reportArgumentType]
)


@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))


agent = create_agent(
    model=basic_model,  # Default model
    tools=[],
    middleware=[dynamic_model_selection],
)

In [4]:
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


@tool
def book_hotel(location: str) -> str:
    """Book a hotel room at a location."""
    return f"Booked a room at {location}."


model_with_tools = model.bind_tools(
    [get_weather, book_hotel], tool_choice="get_weather"
)

In [6]:
from pydantic import BaseModel, Field
from rich import print as rprint


class Movie(BaseModel):
    """A movie with details."""

    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")


model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("以 JSON 格式输出《英雄本色》的信息")
rprint(response)

Movie(title='英雄本色', year=1986, director='吴宇森')

In [7]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("以 JSON 格式输出《英雄本色》的信息")
rprint(response)

{
    'raw': AIMessage(
        content='{\n  "title": "英雄本色",\n  "original_title": "A Better Tomorrow",\n  "year": 1986,\n  
"director": "吴宇森",\n  "country": "中国香港",\n  "language": "粤语",\n  "genre": [\n    "动作",\n    "犯罪",\n   
"剧情"\n  ],\n  "runtime_minutes": 95,\n  "rating": {\n    "imdb": 7.7,\n    "douban": 8.7\n  },\n  "main_cast": 
[\n    {\n      "actor": "周润发",\n      "role": "Mark Lee（小马哥）"\n    },\n    {\n      "actor": "狄龙",\n    
"role": "宋子豪"\n    },\n    {\n      "actor": "张国荣",\n      "role": "宋子杰"\n    }\n  ],\n  "plot_summary": 
"退隐江湖的伪钞集团骨干宋子豪试图重新做人，却因弟弟宋子杰对黑帮的仇恨以及昔日兄弟Mark的复仇行动而再次卷入纷争。三人
之间的情义、背叛与救赎在枪林弹雨中展开。",\n  "awards": [\n    "第6届香港电影金像奖最佳影片",\n    
"第6届香港电影金像奖最佳男主角（周润发）"\n  ],\n  "notes": 
"该片被视为香港黑帮英雄片的经典之作，对后来的华语电影产生了深远影响。"\n}',
        additional_kwargs={'parsed': Movie(title='英雄本色', year=1986, director='吴宇森'), 'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 318,
                'prompt_tokens': 22,
                'total_tokens': 340,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
            },
            'model_provider': 'openai',
            'model_name': 'qwen3-max',
            'system_fingerprint': None,
            'id': 'chatcmpl-0dc600b8-ff43-4052-bc44-1420fc821b17',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--40bd519a-905d-4a25-b532-c3ed333aba7f-0',
        usage_metadata={
            'input_tokens': 22,
            'output_tokens': 318,
            'total_tokens': 340,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {}
        }
    ),
    'parsed': Movie(title='英雄本色', year=1986, director='吴宇森'),
    'parsing_error': None
}

In [ ]:
from langchain.agents.structured_output import ToolStrategy


agent = create_agent(model=model, tools=[], response_format=ToolStrategy(Movie))

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "提取《英雄本色》的信息",
            }
        ]
    }
)

result["structured_response"]

Movie(title='英雄本色', year=1986, director='吴宇森')

In [18]:
result

{'messages': [HumanMessage(content='提取《英雄本色》的信息', additional_kwargs={}, response_metadata={}, id='5bfebe2c-0e43-4aa3-9a02-1298cf9ac09f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 315, 'total_tokens': 359, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen3-max', 'system_fingerprint': None, 'id': 'chatcmpl-1c8018ce-5475-4542-a4d8-d604be54f714', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--122e06be-92fd-4be9-bccc-bedb3ac1ac7c-0', tool_calls=[{'name': 'Movie', 'args': {'title': '英雄本色', 'year': 1986, 'director': '吴宇森'}, 'id': 'call_38defb47082444a0876c2696', 'type': 'tool_call'}], usage_metadata={'input_tokens': 315, 'output_tokens': 44, 'total_tokens': 359, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  ToolMessage(content="Returning structured respo